In [326]:
import plotly.graph_objects as go
import json
import pandas as pd

In [327]:
base_path = "../prompt_structure_results"
score_path = "../prompt_eval_results"
split = "test"
template = "Instruct-Query"
use_lang_specific_prompts=False
k = 10
models = [#"minishlab__potion-base-8M",
          "google__embeddinggemma-300m",          
          #"intfloat__multilingual-e5-small",
          #"BAAI__bge-m3",
          "intfloat__multilingual-e5-large-instruct",
          "Qwen__Qwen3-Embedding-0.6B",
          "microsoft__harrier-oss-v1-0.6b",
          ]

dataset = "mteb__reddit-clustering" #"mteb__ARCChallenge" #"mteb__tatoeba-bitext-mining:fin-eng" #"mteb__ARCChallenge"#"mteb__multi-hatecheck:eng" #"mteb__ARCChallenge" #"mteb__tatoeba-bitext-mining:ara-eng" #"mteb__multi-hatecheck:eng" #"mteb__reddit-clustering" #"mteb__stsbenchmark-sts" #"mteb__tatoeba-bitext-mining:fin-eng"
score= "V-score" # "ndcg@10"#"F1" #"Accuracy" #"V-score" # "average_precision" 
subsplit=""
if dataset == "mteb__reddit-clustering":
    subsplit="0"
path = lambda model: f"{base_path}/{model}/{dataset}/{split}/{template}_template/"
path_scores = lambda model: f"{score_path}/{model}/{dataset}/{split}/{template}_template/"

In [328]:

def construct_df(model, show=False):
    scores_path= path_scores(model)+f"results@{k}.json"
    with open(scores_path) as f:
        scores = json.load(f)
    with open(path(model)+f"prompt_geometry{subsplit}.json") as f:
        data1 = json.load(f)
    if show:
        print(data1.keys())
        #print(data1)
    print(scores)
    df_scores = pd.DataFrame.from_dict(scores).T#, orient="index", columns=["score"])
    #columns= [f"prompt{i}" for i in range(len(scores.values()))])
    #df_scores = df_scores.reset_index().rename(columns={"index": "prompt_text"})#, "mean":"score_mean", "std":"score_std"})
    if show: display(df_scores.head())
    #df_scores["score_mean"] = pd.to_numeric(df_scores["score_mean"])
    df_angle = pd.DataFrame.from_dict(data1).T
    #display(df_scores.head())
    if show: display(df_angle.head())
    df = df_scores.merge(df_angle, on='prompt_text')
    #prompt_dict = prompts(dataset)
    #df["prompt_label"] = df["prompt_text"].apply(lambda prompt: prompt_dict[prompt])
    if show: display(df.head())
    return df

#_ = construct_df(models[0], show=False)

In [329]:

def plot(df, x, y="score", colors=None, sizes=None, title="", legend_title=None):
    if colors is None:
        colors = y
    if sizes is None:
        sizes = y
    
    # legend title that explains formatting
    if legend_title is None:
        legend_title = f"colors:{colors}, size:{sizes}"

    # Normalize scores for marker size
    min_size, max_size = 10, 30
    try:
        # parse the value from dictionary
        df["sizes"] = df[sizes].apply(lambda d: float(d['mean']))
        ranks = df["sizes"].rank(method='average')
    except:   # for non-dict format: i.e. prompt_label or score
        ranks = df[sizes].rank(method='average')
    marker_sizes = (
        (ranks - ranks.min()) / (ranks.max() - ranks.min()) * (max_size - min_size) + min_size
    )

    y_vals = df[y].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d))
    #y_err = df[y].apply(lambda d: float(d['std']) if isinstance(d, dict) else float(d[1]) if isinstance(d, list) else 0.0)
    y_err = []
    for line in df[y]:
        if isinstance(line, dict):
            if "std" in line.keys():
                y_err.append(float(line["std"]))
            elif "confidence_interval" in line.keys():
                y_err.append(float(max(line["confidence_interval"])))
            else:
                y_err.append(0.0)
        else:
            y_err.append(0.0)
        

    x_vals = df[x].apply(lambda d: float(d['mean']))
    x_err  = df[x].apply(lambda d: float(d['std']))


    # Create figure
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            mode='markers',
            x=x_vals,
            y=y_vals,
            error_x=dict(type='data', array=x_err, visible=True, color='lightgray'),  # optional std bars
            error_y=dict(type='data', array=y_err, visible=True, color='lightgray'),
            marker=dict(
                size=marker_sizes,
            #    colorscale='Cividis',
                color=df[colors].apply(lambda d: float(d['mean']) if isinstance(d, dict) else float(d[0]) if isinstance(d, list) else float(d)),
                colorbar=dict(title=f"C:{colors} S:{sizes}"),
                showscale=True,
            ),
            text=df['prompt_text'],
            hovertemplate=(
                '<b>Prompt:</b> %{text}<br>'
                '<b>X (distance):</b> %{x:.4f}<br>'
                '<b>Y (score):</b> %{y:.4f}<br>'
            ),
        ),
    )


    # Add one trace per alpha value

    fig.update_layout(
        title = title,
        xaxis_title=x,#'Cos-distance compared to Q-A line',
        yaxis_title=y,#'Prompt performance',
        height=600,
        width=1000,
        template="none",
    )
    fig.update_layout(legend_title_text=legend_title)


    fig.show()

In [330]:
dfs = {}
for m in models:
    try:
        dfs[m] = construct_df(m)
    except Exception as e:
        print(f"Cannot construct results for {m}")
        print(e)

{'prompt0': {'prompt_text': 'NO_PROMPT', 'V-score': {'mean': 0.21779613427855615, 'standard_error': 0.0045876905576535485, 'confidence_interval': [0.20619352745524497, 0.22388310447647145]}, 'AMI': {'mean': 0.214978782308443}, 'Accuracy': {'mean': 0.8102585487906588, 'standard_error': 0.004828089388384764, 'confidence_interval': [0.8004086646098246, 0.8196895969893535]}, 'F1': {'mean': 0.8102585487906588}}, 'prompt1': {'prompt_text': 'EMPTY', 'V-score': {'mean': 0.3168466302121726, 'standard_error': 0.005072826007129625, 'confidence_interval': [0.30573007175136774, 0.32391578297094453]}, 'AMI': {'mean': 0.31439999404733143}, 'Accuracy': {'mean': 0.8205448985265499, 'standard_error': 0.0043577913888036834, 'confidence_interval': [0.8117225270102366, 0.8291856997294252]}, 'F1': {'mean': 0.8205448985265499}}, 'prompt2': {'prompt_text': 'Identify categories in user passages.', 'V-score': {'mean': 0.3751928117693042, 'standard_error': 0.005352289722164464, 'confidence_interval': [0.36333810

In [332]:

for m in dfs.keys():
    df = dfs[m]
    #print(df.columns)
    plot(df, "displacement", y=score, title=f"{dataset}: {m}: Does more movement(x) mean better score(y)")
    #plot(df, "sim_improvement", y=score, title=f"{dataset}: {m}: Does sim-improvement (x) actually mean better score? (y)")
    #plot(df, "chord_similarity", y=score, title=f"{dataset}: {m}: Fraction of change toward Answer (x) vs. performance(y).")
    #plot(df, "knn_retention", y=score,title=f"{dataset}: {m}: Neighborhood retention (x) vs. performance (y)")
    #plot(df, "orthogonal_magnitude", y=score, title=f"{dataset}: {m}:")
    #plot(df, "hard_neg_angulation", y="chord_similarity", colors=score, sizes=score, title=f"{dataset}: {m}: Angle from false negs vs. evaluation score")
    #plot(df, "hard_neg_sim_change", y=score, title=f"{dataset}: {m}: Movement away from false negs vs. evaluation score")#, sizes="sim_improvement", colors="sim_improvement")
    
	# combinations
    #plot(df, "hard_neg_sim_change", y="sim_improvement", sizes="score", colors="knn_retention",title=f"{dataset}: {m}:")
    #plot(df, "hard_neg_angulation", y="sim_improvement", sizes="score", colors="score",title=f"{dataset}: {m}: Angle from false negs vs. sim improvement")
    #plot(df, "displacement", y="sim_improvement", colors="knn_retention", sizes="knn_retention", title=f"{dataset}: {m}: Movement (x) vs. similarity improvement(y).")
    #plot(df, "parallel_fraction", y="sim_improvement", title=f"{dataset}: {m}: Fraction of change toward Answer (x) vs. similarity gains.")
    #plot(df, "displacement", y="parallel_fraction", sizes="score", colors="prompt_label", title=f"{dataset}: {m}: Does more movement(x) mean better score(y)")


    # this is interesting for sem sim
    #plot(df, "chord_similarity", y="orthogonal_magnitude", colors=score, sizes=score)



| Metric	| Question it answers | Value meanings |
|--------|--------|--------|
|chord_similarity	|Does the prompt push q in the same direction as a?| large = yes, small = no |
|sim_improvement	|Does the prompt make pq closer to a? (the practical question)| pos = prompt did move us closer|
|displacement	|How much does the prompt change the embedding?| large = we moved a lot |
|parallel_magnitude	|How much movement is toward the answer? | negative = away, positive = towards |
|orthogonal_magnitude	|How much movement is sideways (perpendicular to the q→a axis)?| large = a lot, small = a little |
|parallel_fraction	|What fraction of total movement goes toward the answer?| large = a lot towards the answer |
|knn_retention | Does the prompt unify structure? | large = yes, small = no |
|hard_neg_sim | For the closest incorrect, how much did we move away? | small = we moved, large = we did not|

